# 02. 데이터 전처리
Netflix 고객 이탈 예측을 위한 데이터 전처리

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/raw/netflix_customer_churn.csv')
print(f'원본 데이터: {df.shape}')
df.head()

원본 데이터: (5000, 14)


,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


---
## 1. 컬럼 제거
- `customer_id`: 고유 ID, 분석 무관
- `monthly_fee`: subscription_type과 1:1 대응 (Basic=8.99, Standard=13.99, Premium=17.99) → 중복 정보

In [ ]:
# 1:1 대응 확인
print('=== subscription_type과 monthly_fee 대응 확인 ===')
print(df.groupby('subscription_type')['monthly_fee'].unique())

# 제거
df = df.drop(columns=['customer_id', 'monthly_fee'])
print(f'\n제거 후: {df.shape}')
df.head()

=== subscription_type과 monthly_fee 대응 확인 ===
subscription_type
Basic        [8.99]
Premium     [17.99]
Standard    [13.99]
Name: monthly_fee, dtype: object

제거 후: (5000, 12)


,age,gender,subscription_type,watch_hours,last_login_days,region,device,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,51,Other,Basic,14.73,29,Africa,TV,1,Gift Card,1,0.49,Action
1,47,Other,Standard,0.70,19,Europe,Mobile,1,Gift Card,5,0.03,Sci-Fi
2,27,Female,Standard,16.32,10,Asia,TV,0,Crypto,2,1.48,Drama
3,53,Other,Premium,4.51,12,Oceania,TV,1,Crypto,2,0.35,Horror
4,56,Other,Standard,1.89,13,Africa,Mobile,1,Crypto,2,0.13,Action


---
## 2. 파생변수 생성

In [ ]:
# 1) inactive_user_flag: 마지막 로그인 30일 이상이면 비활성 사용자
df['inactive_user_flag'] = (df['last_login_days'] >= 30).astype(int)
print('=== inactive_user_flag 분포 ===')
print(df['inactive_user_flag'].value_counts())
print(f'비활성 사용자 이탈률: {df[df["inactive_user_flag"]==1]["churned"].mean():.2%}')
print(f'활성 사용자 이탈률:   {df[df["inactive_user_flag"]==0]["churned"].mean():.2%}')

=== inactive_user_flag 분포 ===
inactive_user_flag
1    2555
0    2445
Name: count, dtype: int64
비활성 사용자 이탈률: 73.86%
활성 사용자 이탈률:   25.69%


In [ ]:
# 2) estimated_days: 추정 가입 기간 (총 시청시간 / 일 평균 시청시간)
df['estimated_days'] = np.where(
    df['avg_watch_time_per_day'] > 0,
    df['watch_hours'] / df['avg_watch_time_per_day'],
    0
)

# 3) login_inactivity_ratio: 비활성 비율 (마지막 로그인 경과일 / 추정 가입 기간)
#    1.0 이상이면 가입 기간보다 오래 안 들어온 것 → 값이 클수록 비활성
df['login_inactivity_ratio'] = np.where(
    df['estimated_days'] > 0,
    df['last_login_days'] / df['estimated_days'],
    0
)

print('=== 파생변수 통계 ===')
df[['inactive_user_flag', 'estimated_days', 'login_inactivity_ratio']].describe().round(2)

---
## 3. 피처 / 타겟 분리

In [ ]:
X = df.drop(columns=['churned'])
y = df['churned']

cat_cols = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
num_cols = [c for c in X.columns if c not in cat_cols]

print(f'X: {X.shape}, y: {y.shape}')
print(f'\n수치형 ({len(num_cols)}개): {num_cols}')
print(f'범주형 ({len(cat_cols)}개): {cat_cols}')

X: (5000, 14), y: (5000,)

수치형 (8개): ['age', 'watch_hours', 'last_login_days', 'number_of_profiles', 'avg_watch_time_per_day', 'inactive_user_flag', 'estimated_days', 'login_inactivity_ratio']
범주형 (6개): ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']


---
## 4. Train / Test 분리

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train 이탈률: {y_train.mean():.2%}')
print(f'Test  이탈률: {y_test.mean():.2%}')

Train: (4000, 14), Test: (1000, 14)
Train 이탈률: 50.30%
Test  이탈률: 50.30%


---
## 5. 인코딩
모델별로 적합한 인코딩 방식이 다르므로 2가지 버전을 준비합니다.

| 인코딩 | 사용 모델 | 이유 |
|---|---|---|
| One-Hot | Logistic Regression, SVM, MLP | 숫자 크기를 계산에 쓰므로 순서 착각 방지 |
| Label | Decision Tree, RF, XGBoost, LightGBM | 분기만 하므로 크기 무관, OHE 비효율 |

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# === Label Encoding 버전 (트리 모델용) ===
label_encoders = {}
X_train_label = X_train.copy()
X_test_label = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_train_label[col] = le.fit_transform(X_train_label[col])
    X_test_label[col] = le.transform(X_test_label[col])
    label_encoders[col] = le

print(f'Label Encoding 버전: {X_train_label.shape}')
X_train_label.head()

Label Encoding 버전: (4000, 14)


,age,gender,subscription_type,watch_hours,last_login_days,region,device,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre,inactive_user_flag,estimated_days,login_inactivity_ratio
1310,23,0,1,4.03,3,3,2,4,5,1.01,5,0,3.990099,0.751861
2139,20,1,2,2.89,13,5,0,2,2,0.21,6,0,13.761905,0.944637
3293,18,0,0,7.65,49,2,0,2,1,0.15,3,1,51.000000,0.960784
134,19,2,0,5.18,43,2,4,4,4,0.12,4,1,43.166667,0.996139
2619,19,0,2,1.68,6,3,1,0,3,0.24,2,0,7.000000,0.857143


In [ ]:
# === One-Hot Encoding 버전 (LR, SVM, MLP용) ===
X_train_ohe = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# train에 있는 컬럼 기준으로 맞추기
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

print(f'One-Hot Encoding 버전: {X_train_ohe.shape}')
X_train_ohe.head()

One-Hot Encoding 버전: (4000, 31)


,age,watch_hours,last_login_days,number_of_profiles,avg_watch_time_per_day,inactive_user_flag,estimated_days,login_inactivity_ratio,gender_Male,gender_Other,...,payment_method_Crypto,payment_method_Debit Card,payment_method_Gift Card,payment_method_PayPal,favorite_genre_Comedy,favorite_genre_Documentary,favorite_genre_Drama,favorite_genre_Horror,favorite_genre_Romance,favorite_genre_Sci-Fi
1310,23,4.03,3,5,1.01,0,3.990099,0.751861,False,False,...,False,False,False,True,False,False,False,False,True,False
2139,20,2.89,13,2,0.21,0,13.761905,0.944637,True,False,...,False,True,False,False,False,False,False,False,False,True
3293,18,7.65,49,1,0.15,1,51.000000,0.960784,False,False,...,False,True,False,False,False,False,True,False,False,False
134,19,5.18,43,4,0.12,1,43.166667,0.996139,False,True,...,False,False,False,True,False,False,False,True,False,False
2619,19,1.68,6,3,0.24,0,7.000000,0.857143,False,False,...,False,False,False,False,False,True,False,False,False,False


---
## 6. 스케일링

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# OHE 버전 스케일링 (수치형 컬럼만)
X_train_ohe_scaled = X_train_ohe.copy()
X_test_ohe_scaled = X_test_ohe.copy()
X_train_ohe_scaled[num_cols] = scaler.fit_transform(X_train_ohe[num_cols])
X_test_ohe_scaled[num_cols] = scaler.transform(X_test_ohe[num_cols])

print('스케일링 완료 (OHE 버전, 수치형 컬럼만)')
X_train_ohe_scaled[num_cols].describe().round(2)

스케일링 완료 (OHE 버전, 수치형 컬럼만)


,age,watch_hours,last_login_days,number_of_profiles,avg_watch_time_per_day,inactive_user_flag,estimated_days,login_inactivity_ratio
count,4000.00,4000.00,4000.00,4000.00,4000.00,4000.00,4000.00,4000.00
mean,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.66,-0.98,-1.71,-1.43,-0.32,-1.03,-1.71,-5.33
25%,-0.88,-0.70,-0.86,-0.73,-0.28,-1.03,-0.87,0.05
50%,0.02,-0.30,-0.01,-0.02,-0.21,0.97,-0.00,0.30
75%,0.86,0.38,0.85,0.69,-0.06,0.97,0.85,0.42
max,1.70,7.60,1.70,1.39,35.57,0.97,3.14,0.53


---
## 7. 전처리 결과 저장

In [ ]:
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 전처리된 데이터 저장
X_train_label.to_csv('../data/processed/X_train_label.csv', index=False)
X_test_label.to_csv('../data/processed/X_test_label.csv', index=False)
X_train_ohe_scaled.to_csv('../data/processed/X_train_ohe_scaled.csv', index=False)
X_test_ohe_scaled.to_csv('../data/processed/X_test_ohe_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# 전처리 도구 저장 (나중에 Streamlit에서 재사용)
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(label_encoders, '../models/label_encoders.pkl')

print('=== 저장 완료 ===')
print(f'Label 버전  - Train: {X_train_label.shape}, Test: {X_test_label.shape}')
print(f'OHE 버전    - Train: {X_train_ohe_scaled.shape}, Test: {X_test_ohe_scaled.shape}')
print(f'타겟        - Train: {y_train.shape}, Test: {y_test.shape}')

=== 저장 완료 ===
Label 버전  - Train: (4000, 14), Test: (1000, 14)
OHE 버전    - Train: (4000, 31), Test: (1000, 31)
타겟        - Train: (4000,), Test: (1000,)


---
## 전처리 요약

| 단계 | 내용 |
|---|---|
| 제거 | customer_id, monthly_fee (subscription_type과 1:1 대응) |
| 파생변수 | inactive_user_flag, estimated_days, login_inactivity_ratio |
| 분리 | train 80% / test 20% (stratify) |
| 인코딩 | Label (트리 모델용), One-Hot (LR/SVM/MLP용) |
| 스케일링 | StandardScaler (OHE 버전 수치형 컬럼만) |

### 다음 노트북(03_modeling)에서 사용할 데이터
- **트리 모델**: `X_train_label`, `X_test_label`
- **LR, SVM, MLP**: `X_train_ohe_scaled`, `X_test_ohe_scaled`
- **CatBoost**: 원본 `X_train`, `X_test` (인코딩 불필요)